In [1]:
import wrds

# CRSP daily stock file (CIZ format, the same table src/data.py downloads from)
PERMNO = 16731
START, END = "1977-10-01", "1977-10-31"

db = wrds.Connection()
query = f"""
SELECT *
FROM crsp.StkDlySecurityData
WHERE permno = {PERMNO}
    AND dlycaldt BETWEEN '{START}' AND '{END}'
ORDER BY dlycaldt
"""
dsf = db.raw_sql(query, date_cols="dlycaldt")
db.close()

dsf

Loading library list...
Done


,permno,yyyymmdd,dlycaldt,dlydelflg,dlyprc,dlyprcflg,dlycap,dlycapflg,dlyprevprc,dlyprevprcflg,...,dlyvol,dlyclose,dlylow,dlyhigh,dlybid,dlyask,dlyopen,dlynumtrd,dlymmcnt,dlyprcvol
0,16731,19771003,1977-10-03,N,25.5,TR,86700.0,BP,25.125,BA,...,300.0,25.5,25.25,25.5,<NA>,<NA>,<NA>,<NA>,<NA>,7650.0
1,16731,19771004,1977-10-04,N,25.0,TR,85000.0,BP,25.5,TR,...,800.0,25.0,25.0,25.125,<NA>,<NA>,<NA>,<NA>,<NA>,20000.0
2,16731,19771005,1977-10-05,N,24.875,TR,84575.0,BP,25.0,TR,...,500.0,24.875,24.875,25.0,<NA>,<NA>,<NA>,<NA>,<NA>,12437.5
3,16731,19771006,1977-10-06,N,24.875,TR,84575.0,BP,24.875,TR,...,200.0,24.875,24.875,24.875,<NA>,<NA>,<NA>,<NA>,<NA>,4975.0
4,16731,19771007,1977-10-07,N,24.875,TR,84575.0,BP,24.875,TR,...,200.0,24.875,24.875,24.875,<NA>,<NA>,<NA>,<NA>,<NA>,4975.0
5,16731,19771010,1977-10-10,N,24.875,TR,84575.0,BP,24.875,TR,...,200.0,24.875,24.875,24.875,<NA>,<NA>,<NA>,<NA>,<NA>,4975.0
6,16731,19771011,1977-10-11,N,24.875,TR,84575.0,BP,24.875,TR,...,800.0,24.875,24.875,24.875,<NA>,<NA>,<NA>,<NA>,<NA>,19900.0
7,16731,19771012,1977-10-12,N,25.0,TR,85000.0,BP,24.875,TR,...,2100.0,25.0,24.875,25.0,<NA>,<NA>,<NA>,<NA>,<NA>,52500.0
8,16731,19771013,1977-10-13,N,24.875,TR,84575.0,BP,25.0,TR,...,400.0,24.875,24.875,24.875,<NA>,<NA>,<NA>,<NA>,<NA>,9950.0
9,16731,19771014,1977-10-14,Y,0.0,DA,<NA>,DE,24.875,TR,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [2]:
# CRSP delisting record, with the meaning of each flag column from crsp.metaflaginfo
db = wrds.Connection()
delist = db.raw_sql(
    f"SELECT * FROM crsp.StkDelists WHERE permno = {PERMNO}",
    date_cols=["delistingdt", "delnextdt", "delamtdt", "deldlydt"],
)
flag_cols = [
    "deldtprcflg",
    "delactiontype",
    "delstatustype",
    "delreasontype",
    "delpaymenttype",
    "delretmisstype",
    "delnextprcflg",
    "deldistype",
]
flag_values = "', '".join(delist[flag_cols].astype(str).stack().unique())
# flag values repeat across flag types (e.g. NA, D1), so keep the delisting-related types:
# CD action, CS status, DR reason, PT payment, PC price, RM missing return, CI distribution
flags = db.raw_sql(
    f"""
    SELECT flagtype, flagvalue, flagtypedesc, flagdesc
    FROM crsp.metaflaginfo
    WHERE flagvalue IN ('{flag_values}')
        AND flagtype IN ('CD', 'CS', 'DR', 'PT', 'PC', 'RM', 'CI')
    ORDER BY flagtype
    """
)
db.close()

display(delist.T)
flags

Loading library list...
Done


,0
permno,16731
delistingdt,1977-10-13 00:00:00
deldtprc,24.875
deldtprcflg,TR
delactiontype,MER
delstatustype,FPAY
delreasontype,UNAV
delpaymenttype,CASH
delpermno,0
delpermco,0


,flagtype,flagvalue,flagtypedesc,flagdesc
0,CD,MER,Delisting Corporate Action Type,Merger
1,CI,D1,Distribution Impact,One Delisting Action
2,CI,NA,Distribution Impact,Not Applicable
3,CS,FPAY,Delist Completion Status Type,Final Payment
4,CS,UNAV,Delist Completion Status Type,Unavailable
5,DR,UNAV,Delisting Reason Type,Unavailable
6,PC,DA,Price Flag - Multiple - See Definition,Delisting Amount (no Delisting Price)
7,PC,NA,Price Flag - Multiple - See Definition,Not Applicable
8,PC,TR,Price Flag - Multiple - See Definition,Closing Trade Price
9,PT,CASH,Delisting Payment Summary Type,Cash
